Feirando - Data Cleaning and Validation

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path.home() / "Feirando" / "datasets"  
CLEAN = Path.home() / "Feirando" / "clean"    
CLEAN.mkdir(exist_ok=True)                   

Import the csv files, parse the date columns as dates, converts the zip codes to string because it starts with a 0

In [2]:
zip_str = {"customer_zip_code_prefix": str,
           "seller_zip_code_prefix": str,
           "geolocation_zip_code_prefix": str}

orders = pd.read_csv(RAW / "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"])

order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"])

order_reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"])

order_payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")

customers = pd.read_csv(RAW / "olist_customers_dataset.csv", dtype=zip_str)

geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv", dtype=zip_str)

products = pd.read_csv(RAW / "olist_products_dataset.csv")

sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv", dtype=zip_str)

category_translation = pd.read_csv(RAW / "product_category_name_translation.csv",
    encoding="utf-8-sig")

Data check

In [6]:
expected = {"customers": 99441, "geolocation": 1000163, "order_items": 112650,
            "order_payments": 103886, "order_reviews": 99224, "orders": 99441,
            "products": 32951, "sellers": 3095, "category_translation": 71}

tables = {"customers": customers, "geolocation": geolocation, "order_items": order_items,
          "order_payments": order_payments, "order_reviews": order_reviews, "orders": orders,
          "products": products, "sellers": sellers, "category_translation": category_translation}

for name, df in tables.items():
    status = "OK" if len(df) == expected[name] else "MISMATCH"
    print(f"{name:22} {len(df):>9,}  {status}")

print(orders.dtypes)

customers                 99,441  OK
geolocation            1,000,163  OK
order_items              112,650  OK
order_payments           103,886  OK
order_reviews             99,224  OK
orders                    99,441  OK
products                  32,951  OK
sellers                    3,095  OK
category_translation          71  OK
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [7]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [8]:
orders['order_status'].value_counts()

delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_status, dtype: int64

In [9]:
dq_log = []

def log(table, issue, rows, rule, reason):
    dq_log.append({"table": table, "issue": issue, "rows_affected": rows,
                   "rule": rule, "reason": reason})

log("orders", "Status counts check", 0,
    "No change; 8 statuses match profiling", "Confirms load is complete before cleaning")

In [10]:
no_date = orders[(orders["order_status"] == "delivered") &
                 (orders["order_delivered_customer_date"].isna())]
print(len(no_date))
no_date

8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [11]:
orders["in_delivery_scope"] = (
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
)

print(orders["in_delivery_scope"].value_counts())

log("orders", "Delivered orders with no customer delivery date", 8,
    "Keep rows; exclude from delivery analysis via in_delivery_scope flag",
    "Only one field missing; carrier dates still valid for seller-handover analysis (H3)")

True     96470
False     2971
Name: in_delivery_scope, dtype: int64


Added a flag to indicate whether an order is in the delivery scope. 
This flag is set to True for orders that have a status of "delivered" and 
a non-null customer delivery date, and False otherwise. 
This allows for easier filtering of orders that are relevant for delivery analysis.

In [14]:
orders[orders["in_delivery_scope"] == False]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,in_delivery_scope
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,False
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaT,2018-06-28,False
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaT,NaT,2018-08-21,False
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaT,NaT,2017-10-03,False
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaT,2018-02-07,False
...,...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01,False
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaT,NaT,2018-02-06,False
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27,False
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaT,NaT,2017-09-15,False


In [20]:
odd = orders[(orders["order_status"] != "delivered") &
             (orders["order_delivered_customer_date"].notna())]

print(len(odd))
odd[["order_id", "order_status", "order_purchase_timestamp",
     "order_delivered_customer_date", "order_estimated_delivery_date"]]

6


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-11-09 14:53:50,2016-12-08


In [23]:
no_carrier = orders[(orders["order_status"] == "delivered") &
                    (orders["order_delivered_carrier_date"].isna())]

print(len(no_carrier))
no_carrier[["order_id", "order_delivered_carrier_date",
            "order_delivered_customer_date", "in_delivery_scope"]]

2


,order_id,order_delivered_carrier_date,order_delivered_customer_date,in_delivery_scope
73222,2aa91108853cecb43c84a5dc5b277475,NaT,2017-11-20 19:44:47,True
92643,2d858f451373b04fb5c984a1cc2defaf,NaT,NaT,False


In [24]:
log("orders", "Delivered orders with no carrier date", 2,
    "Keep rows; 1 already out of delivery scope (overlap with Issue 2), 1 stays in scope; exclude both from seller-vs-carrier attribution (H3)",
    "Customer on-time check needs only delivered and estimated dates; carrier date only needed to split seller vs carrier delay")

In [25]:
no_approval = orders[orders["order_approved_at"].isna()]

print(len(no_approval))
no_approval["order_status"].value_counts()

160


canceled     141
delivered     14
created        5
Name: order_status, dtype: int64

In [26]:
log("orders", "Orders with no payment approval date", 160,
    "No change. 141 canceled + 5 created = normal (payment never cleared). 14 delivered = logging gap; stay in delivery scope, exclude from any approval-based timing",
    "On-time check does not use approval date; delivered orders must have been paid")

In [29]:
orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: order_purchase_timestamp, dtype: int64

Added a flag in_window for the period we are considering for analysis which is Jan 2017 to August 2018.

In [30]:
ts = orders["order_purchase_timestamp"]
orders["in_window"] = (ts >= "2017-01-01") & (ts < "2018-09-01")

print(orders["in_window"].value_counts())

log("orders", "Sparse months outside analysis window", 349,
    "Keep rows; flag in_window = purchases Jan 2017 to Aug 2018",
    "2016 launch months and Sep-Oct 2018 too thin; Jan 2017 (800) kept despite <1000 threshold: ~±2pp precision, longer follow-up for H5")

True     99092
False      349
Name: in_window, dtype: int64


Added delay_days and is_late columns which shows how many days was the delivery delayed based only on the promised delivery date and not the time. 

In [33]:
delivered_day = orders["order_delivered_customer_date"].dt.normalize()
promised_day = orders["order_estimated_delivery_date"]

orders["delay_days"] = (delivered_day - promised_day).dt.days.where(orders["in_delivery_scope"])
orders["is_late"] = (orders["delay_days"] > 0).where(orders["in_delivery_scope"])

core = orders[orders["in_delivery_scope"] & orders["in_window"]]
print(len(core))
print(core["is_late"].mean())
print(core["delay_days"].describe())

log("orders", "Late/on-time flag created", 0,
    "delay_days = delivered calendar day - promised day; is_late = delay_days > 0; only for in_delivery_scope rows",
    "Promised date has no time of day, so same-day delivery at 18:00 counts as on time")

96203
0.06788769580990198
count    96203.000000
mean       -11.807969
std         10.088942
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delay_days, dtype: float64


In [41]:
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,in_delivery_scope,in_window,delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,True,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,True,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,True,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,True,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,True,True,-10.0,False


Saving the csv file to my clean floder

In [42]:
orders.to_csv(CLEAN / "orders_clean.csv", index=False)


## TABLE 2: order_items

In [47]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [45]:
dupes = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print(dupes)

log("order_items", "Duplicate order_id + order_item_id", 0,
    "No change", "Grain confirmed: one row per item within an order")

0


Lets check if there are any orders in our orders table which does'nt have items from 
our order_items table

In [ ]:
no_items = orders[~orders["order_id"].isin(order_items["order_id"])]

print(len(no_items))
print(no_items["order_status"].value_counts())
print(no_items["in_delivery_scope"].sum())

775
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: order_status, dtype: int64
0


In [52]:
log("order_items", "Orders with no item rows", 775,
    "No change; none in delivery scope (603 unavailable, 164 canceled, 8 other)",
    "Items removed when seller could not fulfil; contribute 0 to GMV automatically")

In [53]:
bad_limit = order_items["shipping_limit_date"] > "2018-12-31"
print(bad_limit.sum())
order_items[bad_limit][["order_id", "shipping_limit_date", "price"]]

4


,order_id,shipping_limit_date,price
8643,13bdf405f961a6deec817d817f5c6624,2020-02-05 03:30:51,69.99
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,2020-02-03 20:23:22,75.99
85729,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99
85730,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99


In [54]:
order_items["shipping_limit_flag"] = order_items["shipping_limit_date"] > "2018-12-31"
order_items.loc[order_items["shipping_limit_flag"], "shipping_limit_date"] = pd.NaT

print(order_items["shipping_limit_flag"].sum())
print(order_items["shipping_limit_date"].max())

log("order_items", "shipping_limit_date after 2018 (up to Apr 2020)", 4,
    "Keep rows; set date to NaT; flag in shipping_limit_flag",
    "Only the date is impossible; price, seller and delivery data valid")

4
2018-09-18 21:10:15


In [59]:
order_items[order_items["shipping_limit_flag"]]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,shipping_limit_flag
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,NaT,69.99,14.66,True
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,NaT,75.99,14.70,True
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,NaT,99.99,61.44,True
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,NaT,99.99,61.44,True


Now lets check of our price and freight values are reasonable.  
are there any negative values?  Are there any extreme outliers? 
We will check the distributions and then decide if we need to filter any rows.

In [60]:
print(order_items[["price", "freight_value"]].describe())
print((order_items["price"] <= 0).sum())
print((order_items["freight_value"] == 0).sum())

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000
0
383


In [61]:
log("order_items", "Price/freight sanity check", 383,
    "No change; no zero/negative prices; 383 items with freight 0 kept",
    "Freight 0 assumed to be free-shipping promotion (cannot verify); extremes plausible")

In [62]:
sellers_per_order = order_items.groupby("order_id")["seller_id"].nunique()
multi = sellers_per_order[sellers_per_order > 1].index

print(len(multi))
print(core["order_id"].isin(multi).sum())
print(core[core["order_id"].isin(multi)]["is_late"].mean())

1278
1272
0.010220125786163521


There are 1272 orders that have multiple sellers and the late rate is 1.0% which means 
out of 6500 late orders only 13 of them have multi sellers. This is a very small 
fraction of the total late orders and can be ignored for the purpose of this analysis.
Also we don't have the seller-level promised date to check which seller was late,
so we cannot attribute the delay to a specific seller.
So we flag it with multi_seller.

In [64]:
orders["multi_seller"] = orders["order_id"].isin(multi)

print(orders["multi_seller"].sum())

log("order_items", "Orders with items from more than one seller", 1278,
    "Keep rows; flag multi_seller on orders; exclude from seller-level analysis (H3) only",
    "One delivery date per order, so lateness cannot be attributed to one seller; blaming all would be unfair; 1,272 in analysis set, ~13 late")

1278


In [65]:
order_items.to_csv(CLEAN / "order_items_clean.csv", index=False)
orders.to_csv(CLEAN / "orders_clean.csv", index=False)

print(pd.read_csv(CLEAN / "order_items_clean.csv").shape)
print(pd.read_csv(CLEAN / "orders_clean.csv").shape)

(112650, 8)
(99441, 13)


## TABLE 3: order_reviews

In [68]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [66]:
print(order_reviews.duplicated().sum())
print(order_reviews.duplicated(subset=["review_id", "order_id"]).sum())

0
0


In [67]:
log("order_reviews", "Exact duplicate rows", 0,
    "No change; none found",
    "Step 8 '814 duplicates' were repeated review_ids on different orders, not copies; data dictionary to be corrected")

In [72]:
dup_ids = order_reviews[order_reviews["review_id"].duplicated(keep=False)]

print(dup_ids["review_id"].nunique())
dup_ids.sort_values("review_id")[["review_id", "order_id", "review_score",
                                  "review_creation_date"]].head(6)

789


,review_id,order_id,review_score,review_creation_date
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07


In [73]:
multi_rev = order_reviews[order_reviews["order_id"].duplicated(keep=False)]

print(multi_rev["order_id"].nunique())
multi_rev.sort_values(["order_id", "review_answer_timestamp"])[
    ["order_id", "review_id", "review_score", "review_answer_timestamp"]].head(6)

547


,order_id,review_id,review_score,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2018-01-02 10:54:47


There are some orders with multiple review, it could be that the customer changed their
mind and updated the review. For the purpose of this analysis, we will keep the 
latest review for each order.

In [74]:
before = len(order_reviews)

order_reviews = (order_reviews
                 .sort_values("review_answer_timestamp")
                 .drop_duplicates(subset="order_id", keep="last"))

print(before, len(order_reviews))
print(order_reviews["order_id"].duplicated().sum())

log("order_reviews", "Orders with more than one review", 547,
    "Keep only the latest review per order (by review_answer_timestamp)",
    "Latest answer is the shopper's final opinion; one review per order prevents double-counting in H1/H2")

99224 98673
0


In [75]:
no_review = ~orders["order_id"].isin(order_reviews["order_id"])
print(no_review.sum())

core_no_review = ~core["order_id"].isin(order_reviews["order_id"])
print(core_no_review.sum())
print(core[core_no_review]["is_late"].mean())

768
643
0.2379471228615863


In [79]:
order_reviews.to_csv(CLEAN / "order_reviews_clean.csv", index=False)
print(pd.read_csv(CLEAN / "order_reviews_clean.csv").shape)

(98673, 7)


## Table 4: order_payments

In [81]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Lets check if the payment values are consistent with the order item values. We will check if the sum 
of the order item prices and the freight values is equal to the payment values for each order.

In [86]:
paid = order_payments.groupby("order_id")["payment_value"].sum()

order_items["item_total"] = order_items["price"] + order_items["freight_value"]
charged = order_items.groupby("order_id")["item_total"].sum()

recon = pd.concat([paid, charged], axis=1, join="inner")
recon.columns = ["paid", "charged"]
recon["diff"] = recon["paid"] - recon["charged"]

print(len(recon))
print((recon["diff"].abs() > 1).sum())
print(recon["diff"].describe())

98665
249
count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: diff, dtype: float64


In [87]:
log("order_payments", "Payments vs item price + freight", 249,
    "No change; 99.75% of orders reconcile within 1 BRL",
    "Differences consistent with instalment interest (+) and vouchers (-); GMV calculated from item price, not payments")

In [88]:
order_payments.to_csv(CLEAN / "order_payments_clean.csv", index=False)
print(pd.read_csv(CLEAN / "order_payments_clean.csv").shape)

(103886, 5)


## Table 5: customers

In [90]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Lets first check if we have unique customer id's per order


In [89]:
print(customers["customer_id"].is_unique)
print(customers["customer_id"].isin(orders["customer_id"]).all())

True
True


Lets check the repeat buyers

In [91]:
orders_per_shopper = customers.groupby("customer_unique_id")["customer_id"].nunique()

print(len(orders_per_shopper))
print((orders_per_shopper > 1).sum())
print((orders_per_shopper > 1).mean())

96096
2997
0.031187562437562436


In [ ]:
log("customers", "Real shoppers vs customer_id", 0,
    "No change; 96,096 shoppers, 2,997 (3.1%) with more than one order",
    "customer_id is new per order; customer_unique_id used for shopper counts and repeat purchase analysis (H5)")

In [93]:
print(customers["customer_zip_code_prefix"].str.len().value_counts())
print(customers["customer_zip_code_prefix"].str.startswith("0").sum())

5    99441
Name: customer_zip_code_prefix, dtype: int64
23995


In [95]:
print(customers["customer_state"].nunique())
print(customers["customer_state"].value_counts().head(5))

log("customers", "State values", 0,
    "No change; 27 valid states; SP 42%, Southeast 66% of orders",
    "Valid codes; concentration relevant for H4 (cross-state routes)")

27
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: customer_state, dtype: int64


In [96]:
customers.to_csv(CLEAN / "customers_clean.csv", index=False)
print(pd.read_csv(CLEAN / "customers_clean.csv", dtype={"customer_zip_code_prefix": str}).shape)

(99441, 5)


## Table 6: geolocation

In [98]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [99]:
in_brazil = (geolocation["geolocation_lat"].between(-33.75, 5.27) &
             geolocation["geolocation_lng"].between(-73.99, -34.79))

print((~in_brazil).sum())
print(geolocation[~in_brazil][["geolocation_zip_code_prefix", "geolocation_lat",
                               "geolocation_lng", "geolocation_state"]].head())

42
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
387565                       18243        28.008978       -15.536867   
513631                       28165        41.614052        -8.411675   
513643                       28155       -34.586422       -58.732101   
513754                       28155        42.439286        13.820214   
514429                       28333        38.381672        -6.328200   

       geolocation_state  
387565                SP  
513631                RJ  
513643                RJ  
513754                RJ  
514429                RJ  


In [100]:
before = len(geolocation)
zips_before = geolocation["geolocation_zip_code_prefix"].nunique()

geolocation = geolocation[in_brazil]

print(before - len(geolocation))
print(zips_before - geolocation["geolocation_zip_code_prefix"].nunique())

log("geolocation", "Points outside Brazil", 42,
    "Remove rows",
    "Geocoding matched same-named towns abroad; true coordinates unknown; each zip has other valid points")

42
5


There are 50 rows of location values per zip. we are going to gorup them per zip.

In [ ]:
geo_zip = (geolocation
           .groupby("geolocation_zip_code_prefix")
           .agg(lat=("geolocation_lat", "median"),
                lng=("geolocation_lng", "median"),
                state=("geolocation_state", "first"))
           .reset_index())

print(len(geo_zip))
print(geo_zip.head())

log("geolocation", "~50 points per zip", len(geolocation) - len(geo_zip),
    "Aggregate to one point per zip: median lat/lng",
    "Joins need one row per zip, otherwise order rows multiply ~50x; median resists stray points")

19010
  geolocation_zip_code_prefix        lat        lng state
0                       01001 -23.550381 -46.634027    SP
1                       01002 -23.548551 -46.635072    SP
2                       01003 -23.548977 -46.635313    SP
3                       01004 -23.549535 -46.634771    SP
4                       01005 -23.549612 -46.636532    SP


In [104]:
cust_missing = ~customers["customer_zip_code_prefix"].isin(geo_zip["geolocation_zip_code_prefix"])
sell_missing = ~sellers["seller_zip_code_prefix"].isin(geo_zip["geolocation_zip_code_prefix"])

print(cust_missing.sum(), cust_missing.mean())
print(sell_missing.sum(), sell_missing.mean())

279 0.0028056837722870848
7 0.002261712439418417


In [105]:
log("geolocation", "Customer/seller zips with no location", 286,
    "No change; 279 customers and 7 sellersexcluded from distance bands only",
    "Zips absent from geolocation; same-state vs cross-state test uses state columns and is unaffected")

In [106]:
geo_zip.to_csv(CLEAN / "geolocation_clean.csv", index=False)
print(pd.read_csv(CLEAN / "geolocation_clean.csv",
                  dtype={"geolocation_zip_code_prefix": str}).shape)

(19010, 4)


## Table 6: products, sellers, category translation

In [107]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


The columns product_name_lenght and product_description_lenght are misspelled in the original dataset. 
They are renamed to product_name_length and product_description_length for clarity.

In [108]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"})

print(products.columns.tolist())

log("products", "Misspelled column names", 0,
    "Rename 'lenght' to 'length' in 2 columns",
    "Prevents typos and confusion in later SQL, Python and Tableau work")

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [112]:
no_cat = products["product_category_name"].isna()

print(no_cat.sum())
print(products[no_cat][["product_description_length", "product_photos_qty"]].isna().all().tolist())

items_no_cat = order_items["product_id"].isin(products.loc[no_cat, "product_id"])
print(items_no_cat.sum(), items_no_cat.mean())

610
[True, True]
1603 0.014229915667998225


Lets fill the products with no category with unknown category, so that they can be included in the analysis. 
Beacuse 1.4% of the products have no category, and they are not all missing description or photos,
it is better to keep them in the analysis with a category of "unknown" rather than dropping them. 
This will allow for a more comprehensive analysis of the product data.

In [118]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

print(products["product_category_name"].isna().sum())
print((products["product_category_name"] == "unknown").sum())

log("products", "Products with no category or listing details", 610,
    "Keep rows; label category 'unknown'; description/photos stay missing",
    "1,603 sold items (1.4%) use these products; deleting would break joins and drop real orders")

0
610


In [119]:
missing_en = set(products["product_category_name"]) - set(category_translation["product_category_name"])
print(missing_en)

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown'}


There are 3 products category names in the tranlation table which do not have an english tranlastion. 
lets add them ourselves.

In [120]:
new_rows = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown"],
    "product_category_name_english": ["gaming_pc", "portable_kitchen_appliances", "unknown"]})

category_translation = pd.concat([category_translation, new_rows], ignore_index=True)

print(len(category_translation))
print(set(products["product_category_name"]) - set(category_translation["product_category_name"]))

log("product_category_translation", "Categories with no English name", 3,
    "Add 3 rows: pc_gamer, portateis_cozinha..., unknown",
    "Every product needs an English category for analysis and dashboard; translations written by analyst")

74
set()


Lets add the english category to the products table, so that it can be used in later analysis and dashboards
 without needing to join to the translation table every time. 

In [123]:
before = len(products)

products = products.merge(category_translation, on="product_category_name", how="left")

print(before, len(products))
print(products["product_category_name_english"].isna().sum())

log("products", "English category added", 0,
    "Add product_category_name_english via left join to translation table",
    "Avoids a join in every later step; translation table is small and fixed")

32951 32951
0


In [124]:
products.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [125]:
print((products["product_weight_g"] == 0).sum())
print(products["product_weight_g"].isna().sum())
print(products[products["product_weight_g"] == 0]["product_category_name_english"].value_counts())

4
2
bed_bath_table    4
Name: product_category_name_english, dtype: int64


In [126]:
import numpy as np

products.loc[products["product_weight_g"] == 0, "product_weight_g"] = np.nan

print((products["product_weight_g"] == 0).sum())
print(products["product_weight_g"].isna().sum())

log("products", "Weight 0 g (4) and missing weight/dimensions (2)", 6,
    "Set 0 g to missing; keep all rows; 2 already-missing left as is",
    "Physical products cannot weigh 0 g (all bed_bath_table); 0 would distort weight averages")

0
6


In [128]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


Now lest check if every seller id is unique and if every seller id in the order_items table is present in the sellers table.
Also if the zip codes are in order, also check the number of states. 
And the share of orders between the states.

In [127]:
print(sellers["seller_id"].is_unique)
print(sellers["seller_id"].isin(order_items["seller_id"]).all())
print(sellers["seller_zip_code_prefix"].str.len().value_counts())
print(sellers["seller_state"].nunique())
print(sellers["seller_state"].value_counts(normalize=True).head(3))

True
True
5    3095
Name: seller_zip_code_prefix, dtype: int64
23
SP    0.597415
PR    0.112763
MG    0.078837
Name: seller_state, dtype: float64


In [129]:
products.to_csv(CLEAN / "products_clean.csv", index=False)
sellers.to_csv(CLEAN / "sellers_clean.csv", index=False)
category_translation.to_csv(CLEAN / "category_translation_clean.csv", index=False)

print(pd.read_csv(CLEAN / "products_clean.csv").shape)
print(pd.read_csv(CLEAN / "sellers_clean.csv", dtype={"seller_zip_code_prefix": str}).shape)
print(pd.read_csv(CLEAN / "category_translation_clean.csv").shape)

(32951, 10)
(3095, 4)
(74, 2)


## Validation

In [130]:
checks = {
    # 1. Row counts: only reviews and geolocation should have changed
    "orders rows = 99,441":        len(orders) == 99441,
    "order_items rows = 112,650":  len(order_items) == 112650,
    "payments rows = 103,886":     len(order_payments) == 103886,
    "reviews rows = 98,673":       len(order_reviews) == 98673,
    "customers rows = 99,441":     len(customers) == 99441,
    "products rows = 32,951":      len(products) == 32951,
    "sellers rows = 3,095":        len(sellers) == 3095,

    # 2. No duplicate keys
    "orders: order_id unique":     orders["order_id"].is_unique,
    "reviews: one per order":      order_reviews["order_id"].is_unique,
    "geo: one row per zip":        geo_zip["geolocation_zip_code_prefix"].is_unique,

    # 3. Join keys: every child row finds its parent
    "items -> orders":     order_items["order_id"].isin(orders["order_id"]).all(),
    "payments -> orders":  order_payments["order_id"].isin(orders["order_id"]).all(),
    "reviews -> orders":   order_reviews["order_id"].isin(orders["order_id"]).all(),
    "orders -> customers": orders["customer_id"].isin(customers["customer_id"]).all(),
    "items -> products":   order_items["product_id"].isin(products["product_id"]).all(),
    "items -> sellers":    order_items["seller_id"].isin(sellers["seller_id"]).all(),
    "every category has English": products["product_category_name_english"].notna().all(),

    # 4. Totals unchanged: GMV must match the raw file
    "GMV matches raw": round(order_items["price"].sum(), 2) ==
                       round(pd.read_csv(RAW / "olist_order_items_dataset.csv")["price"].sum(), 2),
}

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

PASS  orders rows = 99,441
PASS  order_items rows = 112,650
PASS  payments rows = 103,886
PASS  reviews rows = 98,673
PASS  customers rows = 99,441
PASS  products rows = 32,951
PASS  sellers rows = 3,095
PASS  orders: order_id unique
PASS  reviews: one per order
PASS  geo: one row per zip
PASS  items -> orders
PASS  payments -> orders
PASS  reviews -> orders
PASS  orders -> customers
PASS  items -> products
PASS  items -> sellers
PASS  every category has English
PASS  GMV matches raw


In [131]:
log("all tables", "Final validation", 0,
    "18 cross-table checks passed",
    "Row counts, unique keys, join keys in both directions, GMV unchanged vs raw")

dq = pd.DataFrame(dq_log)
print(len(dq))
print(dq["issue"].duplicated().sum())

dq.to_csv(CLEAN / "data_quality_log.csv", index=False)
dq

26
0


,table,issue,rows_affected,rule,reason
0,orders,Status counts check,0,No change; 8 statuses match profiling,Confirms load is complete before cleaning
1,orders,Delivered orders with no customer delivery date,8,Keep rows; exclude from delivery analysis via ...,Only one field missing; carrier dates still va...
2,orders,Canceled orders with a customer delivery date,6,Keep rows; stay out of delivery scope (status ...,Likely canceled after delivery (refusal/return...
3,orders,Delivered orders with no carrier date,2,Keep rows; 1 already out of delivery scope (ov...,Customer on-time check needs only delivered an...
4,orders,Orders with no payment approval date,160,No change. 141 canceled + 5 created = normal (...,On-time check does not use approval date; deli...
5,orders,Sparse months outside analysis window,349,Keep rows; flag in_window = purchases Jan 2017...,2016 launch months and Sep-Oct 2018 too thin; ...
6,orders,Late/on-time flag created,0,delay_days = delivered calendar day - promised...,"Promised date has no time of day, so same-day ..."
7,order_items,Duplicate order_id + order_item_id,0,No change,Grain confirmed: one row per item within an order
8,order_items,Orders with no item rows,775,No change; none in delivery scope (603 unavail...,Items removed when seller could not fulfil; co...
9,order_items,shipping_limit_date after 2018 (up to Apr 2020),4,Keep rows; set date to NaT; flag in shipping_l...,"Only the date is impossible; price, seller and..."
